In [2]:
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KernelDensity
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, AdaBoostClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix

warnings.filterwarnings('ignore')

SEARCH_PATHS = [
    (Path.cwd().parent / "data" / "flows.csv"),
    Path("mirai_flows.csv"),
    Path("Mirai.csv"),
    Path.home() / "Downloads" / "mirai_flows.csv",
    Path.home() / "datasets" / "mirai_flows.csv",
]

CSV_PATH = next((p for p in SEARCH_PATHS if p.exists()), None)
if CSV_PATH is None:
    raise FileNotFoundError(
        "Dataset Mirai no encontrado.\n"
        "Columna objetivo esperada: 'label' con valores 'benign' / nombre_ataque."
    )

df = pd.read_csv(CSV_PATH)
print(f"Shape: {df.shape}")
print(df['label'].value_counts())

Shape: (1006776, 13)
label
1    998983
0      7793
Name: count, dtype: int64


In [3]:
df = pd.read_csv(CSV_PATH)
df['src_ip'] = df['src_ip'].astype(str)

# Estado reclasificado semánticamente (igual que Markov Mirai)
def reclassify_state(row):
    proto  = row['state']
    b_pkts = row['b_pkts']
    avg_ps = row['avg_pkt_size']
    if proto == 'DNS':               return 'DNS_FLOOD'
    if proto in ('HTTP', 'HTTPS'):   return 'HTTP_FLOOD'
    if proto == 'SSH':               return 'OTHER'
    if proto == 'UDP_OTHER':
        if b_pkts == 0 and avg_ps < 100: return 'UDP_SMALL_NORESPONSE'
        elif b_pkts == 0:                return 'UDP_LARGE_NORESPONSE'
        else:                            return 'UDP_BIDIRECTIONAL'
    if proto == 'TCP_OTHER':
        if b_pkts == 0 and avg_ps < 80: return 'TCP_SYN_LIKE'
        elif b_pkts == 0:               return 'TCP_ACK_LIKE'
        else:                           return 'TCP_ESTABLISHED'
    return 'OTHER'
df['state'] = df.apply(reclassify_state, axis=1)

# Subsampleo por clase — igual que Markov Mirai (Hwang Tabla 12)
HWANG_CLASS_TOTAL = {
    'Normal':       68200 + 8525,
    'ACK_Flood':     6600 +  825,
    'DNS_Flood':     4312 +  539,
    'Mirai_CnC':    68200 + 8525,
    'GREIP_Flood':  24712 + 3089,
    'HTTP_Flood':     120 +   15,
    'SYN_Flood':    68200 + 8525,
    'UDP_Flood':    28816 + 3062,
    'VSE_Flood':     4432 +  554,
}

np.random.seed(42)
idx_keep_list = []
for cls_name, n_total in HWANG_CLASS_TOTAL.items():
    idx_cls = np.where(df['class_name'].values == cls_name)[0]
    if len(idx_cls) == 0:
        print(f'  AVISO: clase {cls_name!r} no encontrada')
        continue
    if len(idx_cls) > n_total:
        idx_cls = np.random.choice(idx_cls, n_total, replace=False)
    idx_keep_list.append(idx_cls)
idx_keep = np.sort(np.concatenate(idx_keep_list))
df = df.iloc[idx_keep].reset_index(drop=True)

# Features (igual que Markov Mirai)
FEATURE_COLS = ['n_pkts', 'n_bytes', 'f_pkts', 'f_bytes',
                'b_pkts', 'b_bytes', 'avg_pkt_size', 'duration', 'state']
df['state'] = LabelEncoder().fit_transform(df['state'].astype(str))

X = df[FEATURE_COLS].values.astype(np.float32)
y = df['label'].values.astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# MinMaxScaler aplicado DESPUÉS del split (igual que Autoencoder Mirai)
scaler  = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

CLASSES = sorted(np.unique(y_train))
CLASS_NAMES = {0: 'Benign', 1: 'Malicious'}

print(f"Train: {len(X_train):,} | Test: {len(X_test):,}")
print(f"Features: {len(FEATURE_COLS)}")
print(f"Normal (0): {(y==0).sum():,}  |  Ataque (1): {(y==1).sum():,}")

Train: 129,107 | Test: 32,277
Features: 9
Normal (0): 7,793  |  Ataque (1): 153,591


In [4]:
class KDENaiveBayes(BaseEstimator, ClassifierMixin):
    _estimator_type = "classifier"

    def __init__(self, bandwidth=0.5):
        self.bandwidth = bandwidth

    def fit(self, X, y):
        X, y = np.array(X), np.array(y)
        self.classes_ = np.unique(y)
        self.kdes_, self.priors_ = {}, {}
        for c in self.classes_:
            X_c = X[y == c]
            self.priors_[c] = len(X_c) / len(X)
            self.kdes_[c] = KernelDensity(bandwidth=self.bandwidth).fit(X_c)
        return self

    def predict_proba(self, X):
        X = np.array(X)
        log_p = np.array([
            np.log(self.priors_[c]) + self.kdes_[c].score_samples(X)
            for c in self.classes_
        ]).T
        log_p -= log_p.max(axis=1, keepdims=True)
        p = np.exp(log_p)
        return p / p.sum(axis=1, keepdims=True)

    def predict(self, X):
        return self.classes_[np.argmax(self.predict_proba(X), axis=1)]


classifiers = {
    'A1 - J48'          : DecisionTreeClassifier(criterion='entropy', min_samples_leaf=2, random_state=1),
    'A2 - Meta Pagging' : BaggingClassifier(estimator=DecisionTreeClassifier(min_samples_leaf=2, random_state=1), n_estimators=10, max_samples=1.0, random_state=1),
    'A3 - RandomTree'   : DecisionTreeClassifier(max_features=6, min_samples_leaf=1, random_state=1),
    'A4 - REPTree'      : DecisionTreeClassifier(min_samples_leaf=2, random_state=1),
    'A5 - AdaBoostM1'   : AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1, random_state=1), n_estimators=10, random_state=1, algorithm='SAMME'),
    'A6 - DecisionStump': DecisionTreeClassifier(max_depth=1, random_state=1),
    'A7 - NaiveBayes'   : KDENaiveBayes(bandwidth=0.5),
    'A8 - SVM'          : SVC(kernel='rbf', probability=True, C=1.0, gamma='scale', random_state=1),
}

trained_clfs = {}
for name, clf in classifiers.items():
    print(f"Entrenando {name}...", end=' ', flush=True)
    clf.fit(X_train, y_train)
    trained_clfs[name] = clf
    print("listo.")

Entrenando A1 - J48... listo.
Entrenando A2 - Meta Pagging... listo.
Entrenando A3 - RandomTree... listo.
Entrenando A4 - REPTree... listo.
Entrenando A5 - AdaBoostM1... listo.
Entrenando A6 - DecisionStump... listo.
Entrenando A7 - NaiveBayes... listo.
Entrenando A8 - SVM... listo.


In [5]:
def compute_weka_metrics(y_true, y_pred, classes):
    cm = confusion_matrix(y_true, y_pred, labels=classes)
    tp_rates, fp_rates, supports = [], [], []
    for i in range(len(classes)):
        TP = cm[i, i]
        FN = cm[i, :].sum() - TP
        FP = cm[:, i].sum() - TP
        TN = cm.sum() - TP - FP - FN
        support = cm[i, :].sum()
        tp_rates.append(TP / (TP + FN) if (TP + FN) > 0 else 0.0)
        fp_rates.append(FP / (FP + TN) if (FP + TN) > 0 else 0.0)
        supports.append(support)
    total = sum(supports)
    tp_w = sum(tp * s for tp, s in zip(tp_rates, supports)) / total
    fp_w = sum(fp * s for fp, s in zip(fp_rates, supports)) / total
    return tp_w, fp_w


SEP = '=' * 62
print("Voting Ensemble — Dataset Mirai")
print(SEP)
print(f"{'Clasificador':<22} {'TP Rate':>8} {'FP Rate':>8} {'Accuracy':>10}")
print('-' * 62)

all_probas = []
for name, clf in trained_clfs.items():
    y_pred = clf.predict(X_test)
    acc    = accuracy_score(y_test, y_pred) * 100
    tp, fp = compute_weka_metrics(y_test, y_pred, CLASSES)
    print(f"{name:<22} {tp:>8.3f} {fp:>8.3f} {acc:>9.2f}%")
    if hasattr(clf, 'predict_proba'):
        p = clf.predict_proba(X_test)
        clf_classes = list(clf.classes_)
        p_full = np.zeros((len(X_test), len(CLASSES)))
        for j, c in enumerate(CLASSES):
            if c in clf_classes:
                p_full[:, j] = p[:, clf_classes.index(c)]
        all_probas.append(p_full)

print('-' * 62)
avg_proba  = np.mean(all_probas, axis=0)
y_pred_ens = np.array(CLASSES)[np.argmax(avg_proba, axis=1)]
acc_e      = accuracy_score(y_test, y_pred_ens) * 100
tp_e, fp_e = compute_weka_metrics(y_test, y_pred_ens, CLASSES)
print(f"{'E  - Proposed Model':<22} {tp_e:>8.3f} {fp_e:>8.3f} {acc_e:>9.2f}%")
print(SEP)

Voting Ensemble — Dataset Mirai
Clasificador            TP Rate  FP Rate   Accuracy
--------------------------------------------------------------
A1 - J48                  0.998    0.012     99.82%
A2 - Meta Pagging         0.998    0.012     99.84%
A3 - RandomTree           0.998    0.015     99.83%
A4 - REPTree              0.998    0.014     99.80%
A5 - AdaBoostM1           0.993    0.120     99.28%
A6 - DecisionStump        0.966    0.673     96.56%
A7 - NaiveBayes           0.952    0.952     95.17%
A8 - SVM                  0.996    0.050     99.58%
--------------------------------------------------------------
E  - Proposed Model       0.998    0.021     99.84%
